# 05 — SDK Usage

This notebook demonstrates using the Claude Code SDK programmatically —
running queries, controlling permissions, and building pipelines.

## Learning Objectives
1. Understand the Claude Code SDK interface
2. Run read-only and write-enabled queries
3. Build a simple code review pipeline

## Setup

In [ ]:
import os
import json
import subprocess
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY"
print("Ready.")

## SDK Overview

The Claude Code SDK provides a programmatic interface:

```
Your Script
    ↓
claude --print "prompt" (CLI)
    ↓
Claude Code (same tools as terminal)
    ↓
Claude API
    ↓
Result
```

**Default**: read-only permissions. Write requires explicit opt-in.

## SDK Client Wrapper

Let's build a Python wrapper for the Claude Code CLI.

In [ ]:
class ClaudeCodeSDK:
    """Python wrapper for the Claude Code SDK CLI."""

    def __init__(self, project_dir: str = ".") -> None:
        self.project_dir = project_dir

    def query(
        self,
        prompt: str,
        allowed_tools: list[str] | None = None,
        timeout: int = 120,
    ) -> str:
        """Send a query to Claude Code SDK."""
        cmd = ["claude", "--print"]

        if allowed_tools:
            for tool in allowed_tools:
                cmd.extend(["--allowedTools", tool])

        cmd.append(prompt)

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            cwd=self.project_dir,
            timeout=timeout,
        )

        if result.returncode != 0:
            raise RuntimeError(f"SDK error: {result.stderr}")

        return result.stdout

    def review(self, file_path: str) -> str:
        """Review a file for bugs and issues (read-only)."""
        return self.query(
            f"Review {file_path} for bugs, security issues, and style problems. "
            "Provide findings with severity ratings."
        )

    def generate_tests(self, file_path: str) -> str:
        """Generate tests for a file (needs write access)."""
        return self.query(
            f"Generate comprehensive pytest tests for {file_path}. "
            "Write them to the tests/ directory.",
            allowed_tools=["read", "write", "edit"],
        )

    def explain(self, file_path: str) -> str:
        """Explain what a file does (read-only)."""
        return self.query(
            f"Read {file_path} and explain what it does in 2-3 paragraphs."
        )


print("ClaudeCodeSDK wrapper defined.")
print("Note: This requires the 'claude' CLI to be installed.")

## Simulated SDK (for environments without Claude CLI)

If you don't have the Claude CLI installed, we can simulate the SDK pattern
using the Anthropic API directly.

In [ ]:
import anthropic

client = anthropic.Anthropic()


def simulated_sdk_query(prompt: str, project_context: str = "") -> str:
    """Simulate a Claude Code SDK query using the API."""
    system = "You are a Claude Code assistant reviewing a codebase."
    if project_context:
        system += f"\n\nProject context:\n{project_context}"

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=2048,
        system=system,
        messages=[{"role": "user", "content": prompt}],
    )

    return response.content[0].text


print("Simulated SDK ready.")

## Code Review Pipeline

In [ ]:
# Sample code to review
sample_code = '''
import sqlite3

def get_user(username):
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    query = f"SELECT * FROM users WHERE username = '{username}'"
    cursor.execute(query)
    result = cursor.fetchone()
    return result

def save_user(username, password):
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    cursor.execute(f"INSERT INTO users VALUES ('{username}', '{password}')")
    conn.commit()
'''

review = simulated_sdk_query(
    f"Review this Python code for security vulnerabilities, bugs, and style issues:\n\n```python\n{sample_code}\n```\n\n"
    "Format as a list with severity (Critical/High/Medium/Low) for each finding."
)

print(review)

## Batch Processing Pipeline

In [ ]:
def batch_review(files: dict[str, str]) -> dict[str, str]:
    """Review multiple code snippets."""
    results = {}

    for name, code in files.items():
        print(f"Reviewing: {name}...")
        review = simulated_sdk_query(
            f"Review this code from {name} briefly. List top 3 issues:\n\n"
            f"```python\n{code}\n```"
        )
        results[name] = review

    return results


files_to_review = {
    "auth.py": sample_code,
    "utils.py": "def parse_json(data):\n    return eval(data)\n",
}

reviews = batch_review(files_to_review)

for name, review in reviews.items():
    print(f"\n{'=' * 40}")
    print(f"Review: {name}")
    print(f"{'=' * 40}")
    print(review)

## Permission Comparison

In [ ]:
# Demonstrate the concept of permission levels
permissions = {
    "read-only (default)": {
        "allowed": ["read", "ls", "glob", "grep"],
        "blocked": ["write", "edit", "bash"],
    },
    "with write access": {
        "allowed": ["read", "ls", "glob", "grep", "write", "edit"],
        "blocked": ["bash"],
    },
    "full access": {
        "allowed": ["read", "ls", "glob", "grep", "write", "edit", "bash"],
        "blocked": [],
    },
}

for level, tools in permissions.items():
    print(f"\n{level}:")
    print(f"  ✅ Allowed: {', '.join(tools['allowed'])}")
    if tools['blocked']:
        print(f"  🚫 Blocked: {', '.join(tools['blocked'])}")

## Exercise

1. Extend the `ClaudeCodeSDK` class with a `refactor` method that suggests improvements
2. Build a pipeline that reviews all `.py` files in a directory and produces a summary report
3. Create a hook script that uses `simulated_sdk_query` to detect duplicate code patterns
4. Compare the output quality between read-only review and write-enabled review